# History and reproducibility

The history accessor records mutating calls, named snapshots,
diffs, and exportable provenance.


In [ ]:
import annnet as an


## Record construction history


In [ ]:
G = an.AnnNet(directed=True)
G.history.clear()
G.add_vertices(['A', 'B'])
G.add_edges('A', 'B', edge_id='e1', weight=1.0)
G.history.mark('baseline graph')
G.history.snapshot('baseline')

import polars as pl

print('events:', len(G.history()))
print('last op:', G.history()[-1]['op'])
pl.DataFrame(
    [
        {
            'version': event['version'],
            'operation': event['op'],
            'result': str(event.get('result', '')),
        }
        for event in G.history()
    ]
)


## Compare snapshots


In [ ]:
G.add_vertices('C')
G.add_edges('B', 'C', edge_id='e2', weight=0.5)
G.slices.add('candidate')
G.slices.add_edge_to_slice('candidate', 'e2')
G.history.snapshot('candidate')

diff = G.history.diff('baseline', 'candidate')
import json

print(diff.summary())
print(json.dumps(diff.to_dict(), indent=2))


## Export history


In [ ]:
import os
import tempfile
import json

tmp = tempfile.mkdtemp(prefix='annnet_history_')
json_path = os.path.join(tmp, 'history.json')
n_events = G.history.export(json_path)
with open(json_path, encoding='utf-8') as handle:
    exported = json.load(handle)

print('events written:', n_events)
print('file exists:', os.path.exists(json_path))
print(json.dumps(exported[:4], indent=2))


History is not a workflow engine; it is a compact provenance
layer. It helps notebooks explain how a graph changed and makes
named graph states easy to compare.
